# Copper Data Exploration

This notebook gets familiar with copper-price data before any prediction is attempted. It checks what the data contains, how prices move, and whether the dates are safe to use in a future forecasting experiment.

The main series is Yahoo Finance `HG=F`, a daily copper-futures market price in USD per pound. An optional FRED series, `PCOPPUSDM`, is a separate monthly global copper-price series in USD per metric ton. They have different units and update schedules, so this notebook compares them visually but does not combine them into one price.

## 1. Load Configuration and Libraries

The notebook keeps downloaded data in the repository's `data/` folder. Yahoo data is fetched when its cache is missing. The FRED comparison is turned off by default because it needs either a cached file or a personal FRED API key.

In [18]:
from datetime import datetime, timezone
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv


REPO_ROOT = next(
    path
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "pyproject.toml").exists() and (path / "aieng-forecasting").is_dir()
)
sys.path.insert(0, str(REPO_ROOT / "implementations"))
load_dotenv(REPO_ROOT / ".env", override=False)

from aieng.forecasting.data import DataService, SeriesMetadata
from aieng.forecasting.data.adapters import YFinanceDailyAdapter

COPPER_TICKER = "HG=F"
COPPER_SERIES_ID = "copper_futures_hg_adj_close_usd_lb"
FRED_COPPER_ID = "PCOPPUSDM"
FRED_COPPER_SERIES_ID = "fred_global_copper_price_usd_metric_ton"
YAHOO_CACHE_DIR = REPO_ROOT / "data" / "yfinance"
FRED_CACHE_DIR = REPO_ROOT / "data" / "fred"
YAHOO_START = "2000-01-01"
LOAD_FRED_COMPARISON = True
LOAD_YAHOO_COVARIATES = True
EXPORT_ARTIFACTS = True

print(pd.Series({
    "Primary data": f"Yahoo Finance {COPPER_TICKER} (daily)",
    "Optional comparison": f"FRED {FRED_COPPER_ID} (monthly)",
    "Yahoo cache": YAHOO_CACHE_DIR,
    "FRED comparison enabled": LOAD_FRED_COMPARISON,
    "Yahoo covariates enabled": LOAD_YAHOO_COVARIATES,
    "Export artifacts": EXPORT_ARTIFACTS,
}).to_frame("value"))

                                                                  value
Primary data                                 Yahoo Finance HG=F (daily)
Optional comparison                            FRED PCOPPUSDM (monthly)
Yahoo cache               /home/coder/agentic-forecasting/data/yfinance
FRED comparison enabled                                            True
Yahoo covariates enabled                                           True
Export artifacts                                                   True


### How to Read This Output

This table confirms the notebook settings. Check that the primary data is daily Yahoo Finance `HG=F`, then note whether the optional FRED comparison, related-market data, and artifact export are turned on. The cache paths show where downloaded data is stored.

## 2. Load Copper Price Data

Yahoo Finance is the source for the daily price we will eventually forecast. The adapter first reads its local cache. If the cache is missing, it asks Yahoo Finance for the history and saves it locally for later runs.

A daily closing price is marked as available on the next business day. That prevents a later forecast experiment from accidentally using a day's closing price before that trading day has ended.

In [19]:
def naive_utc_now() -> datetime:
    return datetime.now(tz=timezone.utc).replace(tzinfo=None)


def build_daily_copper_service() -> DataService:
    service = DataService()
    service.register(
        COPPER_SERIES_ID,
        YFinanceDailyAdapter(
            COPPER_TICKER,
            field="Adj Close",
            start=YAHOO_START,
            cache_dir=YAHOO_CACHE_DIR,
        ),
        SeriesMetadata(
            series_id=COPPER_SERIES_ID,
            description="Copper continuous front-month futures adjusted close (Yahoo Finance HG=F)",
            source="Yahoo Finance (HG=F)",
            units="USD per pound",
            frequency="B",
            table_id="yahoo:HG=F:adj-close",
        ),
    )
    return service


try:
    daily_service = build_daily_copper_service()
except (RuntimeError, ValueError) as exc:
    raise RuntimeError(
        "Could not load Yahoo Finance copper data. Check network access or an existing "
        f"cache at {YAHOO_CACHE_DIR}."
    ) from exc

as_of_today = naive_utc_now()
daily_history = daily_service.context(as_of=as_of_today).get_series(COPPER_SERIES_ID)
latest_row = daily_history.iloc[-1]

print(f"Rows loaded: {len(daily_history):,}")
print(f"Date coverage: {daily_history['timestamp'].min().date()} to {daily_history['timestamp'].max().date()}")
print(f"Latest available close: ${latest_row['value']:.2f}/lb on {latest_row['timestamp'].date()}")
print(f"This close became available: {latest_row['released_at'].date()}")

Rows loaded: 6,541
Date coverage: 2000-08-30 to 2026-09-17
Latest available close: $6.62/lb on 2026-09-17
This close became available: 2026-09-18


### How to Read This Output

`Rows loaded` is the number of trading-day observations. `Date coverage` gives the first and last market-close dates. The latest close is the newest copper price in the data, while `became available` is the next business day, when a forecast is allowed to use that close.

## 3. Inspect Data Structure and Quality

Before cleaning, inspect the raw adapter output. A missing weekday can be a normal market holiday, so this notebook reports gaps without treating every missing weekday as bad data.

In [20]:
raw_copper = daily_history.copy()

print("Columns:", raw_copper.columns.tolist())
print("Data types:")
print(raw_copper.dtypes)
print(f"Shape: {raw_copper.shape}")
print(f"Duplicate timestamps: {raw_copper['timestamp'].duplicated().sum()}")
print("Missing values:")
print(raw_copper.isna().sum())
print(f"Date coverage: {raw_copper['timestamp'].min().date()} to {raw_copper['timestamp'].max().date()}")
print(f"Positive prices: {(raw_copper['value'] > 0).sum():,} of {len(raw_copper):,}")

expected_weekdays = pd.bdate_range(raw_copper["timestamp"].min(), raw_copper["timestamp"].max())
observed_dates = pd.DatetimeIndex(raw_copper["timestamp"])
missing_weekdays = expected_weekdays.difference(observed_dates)
print(f"Weekdays without a market close: {len(missing_weekdays):,}")
print("These can be exchange holidays or other normal market closures.")
display(raw_copper.head())

Columns: ['timestamp', 'value', 'released_at']
Data types:
timestamp      datetime64[ns]
value                 float64
released_at    datetime64[ns]
dtype: object
Shape: (6541, 3)
Duplicate timestamps: 0
Missing values:
timestamp      0
value          0
released_at    0
dtype: int64
Date coverage: 2000-08-30 to 2026-09-17
Positive prices: 6,541 of 6,541
Weekdays without a market close: 256
These can be exchange holidays or other normal market closures.


,timestamp,value,released_at
0,2000-08-30,0.8850,2000-08-31
1,2000-08-31,0.8850,2000-09-01
2,2000-09-01,0.8890,2000-09-04
3,2000-09-05,0.9060,2000-09-06
4,2000-09-06,0.9015,2000-09-07


### How to Read This Output

The column list and data types describe the raw table. A good result has no duplicate timestamps, no missing values in important fields, and one positive price per observed date. `Weekdays without a market close` is not automatically an error because exchanges close for holidays. The small table shows the first few raw rows, including the close date and its availability date.

## 4. Clean and Standardize the Time Series

The analysis table has one row per available market close. It is sorted by date, duplicate dates are removed, and prices are converted to numbers. The notebook does not invent prices for market holidays.

### Columns Created Below

- `timestamp`: the date of that row's copper market close.
- `released_at`: when that closing price is allowed to be used. It is usually the next business day, so a forecast cannot see a price before that trading day ends.
- `price_usd_per_lb`: the copper futures closing price, in US dollars per pound.
- `daily_pct_return`: the percentage change from the previous trading day's close. For example, `2.0` means the price rose about 2% that day.
- `daily_log_return`: another mathematical way to measure the same daily change. Forecasting models often use it because changes combine cleanly over time.
- `rolling_volatility_21b_pct`: how jumpy the price has been over the most recent 21 business days, roughly one month. A higher number means more uncertainty.
- `drawdown_pct`: how far the price is below its highest point so far. For example, `-10` means the price is 10% below its previous peak.

In [21]:
copper = raw_copper.loc[:, ["timestamp", "value", "released_at"]].copy()
copper["timestamp"] = pd.to_datetime(copper["timestamp"])
copper["released_at"] = pd.to_datetime(copper["released_at"])
copper["price_usd_per_lb"] = pd.to_numeric(copper.pop("value"), errors="coerce")
copper = copper.dropna(subset=["timestamp", "released_at", "price_usd_per_lb"])
copper = copper[copper["price_usd_per_lb"] > 0]
copper = copper.drop_duplicates("timestamp", keep="last").sort_values("timestamp")
copper = copper.set_index("timestamp").sort_index()

copper["daily_pct_return"] = copper["price_usd_per_lb"].pct_change() * 100
copper["daily_log_return"] = np.log(copper["price_usd_per_lb"] / copper["price_usd_per_lb"].shift(1))
copper["rolling_volatility_21b_pct"] = copper["daily_log_return"].rolling(21).std() * np.sqrt(252) * 100
copper["drawdown_pct"] = (copper["price_usd_per_lb"] / copper["price_usd_per_lb"].cummax() - 1) * 100

assert copper.index.is_monotonic_increasing
assert copper["price_usd_per_lb"].gt(0).all()
print(f"Clean rows: {len(copper):,}")
print(f"Analysis coverage: {copper.index.min().date()} to {copper.index.max().date()}")
display(copper.tail())
display(copper.head())

Clean rows: 6,541
Analysis coverage: 2000-08-30 to 2026-09-17


,released_at,price_usd_per_lb,daily_pct_return,daily_log_return,rolling_volatility_21b_pct,drawdown_pct
timestamp,,,,,,
2026-09-11,2026-09-14,6.4695,0.038659,0.000387,24.449698,-4.909239
2026-09-14,2026-09-15,6.3300,-2.156274,-0.021799,25.494724,-6.959657
2026-09-15,2026-09-16,6.3685,0.608220,0.006064,25.627349,-6.393767
2026-09-16,2026-09-17,6.4315,0.989240,0.009844,25.929242,-5.467777
2026-09-17,2026-09-18,6.6215,2.954211,0.029114,27.151113,-2.675096


,released_at,price_usd_per_lb,daily_pct_return,daily_log_return,rolling_volatility_21b_pct,drawdown_pct
timestamp,,,,,,
2000-08-30,2000-08-31,0.8850,NaN,NaN,NaN,0.000000
2000-08-31,2000-09-01,0.8850,0.000000,0.000000,NaN,0.000000
2000-09-01,2000-09-04,0.8890,0.451978,0.004510,NaN,0.000000
2000-09-05,2000-09-06,0.9060,1.912263,0.018942,NaN,0.000000
2000-09-06,2000-09-07,0.9015,-0.496692,-0.004979,NaN,-0.496692


### How to Read This Output

`Clean rows` and `Analysis coverage` confirm how much usable history remains after cleaning. The first table shows the newest rows; the second shows the oldest. Expect the first return and rolling-volatility values to be blank at the beginning because there is not yet enough earlier history to calculate them.

## 5. Compute Descriptive Statistics

These statistics describe the range and typical size of past daily copper prices. They summarize history; they do not tell us what the next price will be.

In [22]:
price_summary = copper["price_usd_per_lb"].describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]).to_frame("USD per pound")
price_summary.loc["median"] = copper["price_usd_per_lb"].median()
price_summary = price_summary.rename(index={"50%": "50% (median)"})

annual_summary = (
    copper["price_usd_per_lb"]
    .resample("YE")
    .agg(["count", "mean", "median", "std", "min", "max"])
    .rename_axis("year")
)
annual_summary.index = annual_summary.index.year

print("Daily price summary")
display(price_summary.round(3))
print("Annual price summary")
display(annual_summary.tail(10).round(3))

Daily price summary


,USD per pound
count,6541.000
mean,2.933
std,1.275
min,0.604
5%,0.731
25%,2.161
50% (median),3.066
75%,3.743
95%,4.777
max,6.804


Annual price summary


,count,mean,median,std,min,max
year,,,,,,
2017,251,2.805,2.705,0.224,2.481,3.286
2018,250,2.923,2.919,0.203,2.557,3.293
2019,252,2.721,2.688,0.121,2.513,2.976
2020,253,2.803,2.828,0.368,2.119,3.628
2021,252,4.249,4.287,0.283,3.539,4.779
2022,251,4.004,3.839,0.479,3.211,4.929
2023,251,3.853,3.815,0.181,3.542,4.267
2024,252,4.221,4.171,0.303,3.686,5.119
2025,252,4.825,4.810,0.389,3.989,5.795


### How to Read This Output

The daily table summarizes every recorded close. `Mean` is the average, `50% (median)` is the middle value, and the percentiles show where unusually low or high prices begin. The annual table summarizes each of the most recent ten years: `count` is the number of trading days, `std` measures how spread out prices were, and `min` and `max` show that year's range.

## 6. Visualize Copper Price History

The full chart provides long-run context. The recent chart uses a 63-business-day rolling average, which smooths small day-to-day changes and makes the recent direction easier to see. The labels mark the highest and lowest daily closes in the downloaded history.

In [23]:
copper["rolling_mean_63b"] = copper["price_usd_per_lb"].rolling(63).mean()
high_date = copper["price_usd_per_lb"].idxmax()
low_date = copper["price_usd_per_lb"].idxmin()

full_history = px.line(
    copper.reset_index(),
    x="timestamp",
    y="price_usd_per_lb",
    title="Copper futures price history",
    labels={"timestamp": "Date", "price_usd_per_lb": "USD per pound"},
)
full_history.add_annotation(
    x=high_date,
    y=copper.loc[high_date, "price_usd_per_lb"],
    text="Highest close",
    showarrow=True,
)
full_history.add_annotation(
    x=low_date,
    y=copper.loc[low_date, "price_usd_per_lb"],
    text="Lowest close",
    showarrow=True,
)
full_history.show()

recent_start = copper.index.max() - pd.DateOffset(years=1)
recent_copper = copper.loc[recent_start:].reset_index()
recent_history = go.Figure()
recent_history.add_trace(go.Scatter(
    x=recent_copper["timestamp"],
    y=recent_copper["price_usd_per_lb"],
    name="Daily close",
))
recent_history.add_trace(go.Scatter(
    x=recent_copper["timestamp"],
    y=recent_copper["rolling_mean_63b"],
    name="63-business-day average",
))
recent_history.update_layout(
    title="Copper futures price: last year",
    xaxis_title="Date",
    yaxis_title="USD per pound",
)
recent_history.show()

### How to Read These Charts

Both charts use date on the horizontal axis and US dollars per pound on the vertical axis. In the full-history chart, the annotations identify the highest and lowest closes in the downloaded period. In the recent chart, compare the daily-close line with the 63-business-day average: a daily line above the smoother average suggests recent prices are above their roughly three-month trend, while a line below it suggests the opposite.

## 7. Analyze Returns and Volatility

A return is a day-to-day price change. Volatility measures how much those changes vary. Large moves are worth noticing because they show periods where a simple future forecast may be less reliable.

In [24]:
return_frame = copper.dropna(subset=["daily_pct_return", "daily_log_return"]).copy()
large_move_threshold = return_frame["daily_pct_return"].abs().quantile(0.99)
large_moves = return_frame.loc[
    return_frame["daily_pct_return"].abs() >= large_move_threshold,
    ["price_usd_per_lb", "daily_pct_return", "daily_log_return"],
].sort_values("daily_pct_return", key=lambda values: values.abs(), ascending=False)

return_distribution = px.histogram(
    return_frame.reset_index(),
    x="daily_pct_return",
    nbins=80,
    title="Distribution of daily copper price changes",
    labels={"daily_pct_return": "Daily percentage change"},
)
return_distribution.show()

risk_figure = make_subplots(specs=[[{"secondary_y": True}]])
risk_figure.add_trace(
    go.Scatter(x=copper.index, y=copper["rolling_volatility_21b_pct"], name="21-day annualized volatility"),
    secondary_y=False,
)
risk_figure.add_trace(
    go.Scatter(x=copper.index, y=copper["drawdown_pct"], name="Drawdown from prior high"),
    secondary_y=True,
)
risk_figure.update_layout(title="Copper volatility and drawdown")
risk_figure.update_yaxes(title_text="Volatility (%)", secondary_y=False)
risk_figure.update_yaxes(title_text="Drawdown (%)", secondary_y=True)
risk_figure.show()

print(f"Largest-move threshold (top 1%): {large_move_threshold:.2f}%")
display(large_moves.head(10).round(3))

Largest-move threshold (top 1%): 5.64%


,price_usd_per_lb,daily_pct_return,daily_log_return
timestamp,,,
2025-07-31,4.331,-22.253,-0.252
2025-07-08,5.645,13.251,0.124
2008-10-29,2.072,12.490,0.118
2006-05-23,4.076,11.964,0.113
2008-10-10,2.157,-11.035,-0.117
2004-10-13,1.293,-11.012,-0.117
2008-12-08,1.478,9.077,0.087
2025-04-04,4.385,-8.865,-0.093
2008-10-30,1.888,-8.834,-0.092


### How to Read These Outputs

In the histogram, the horizontal axis is a daily percentage change and the vertical axis counts how often changes of that size occurred. Taller bars near zero mean small daily moves are common; bars far from zero are rare, large moves. In the risk chart, the left axis shows 21-business-day volatility, where higher values mean more unsettled daily changes. The right axis shows drawdown: `0%` means a new or matching high, and more-negative values mean a deeper fall from the previous peak. The table lists the ten largest absolute daily changes; the threshold says how large a move had to be to fall in the most extreme 1% of days.

## 8. Assess Trend and Seasonality

Monthly averages remove much of the day-to-day noise. A month-of-year summary can reveal repeated seasonal patterns, but a pattern in the past is not a guarantee for the future.

In [25]:
monthly_prices = copper["price_usd_per_lb"].resample("MS").mean().to_frame("average_price_usd_per_lb")
monthly_prices["monthly_pct_change"] = monthly_prices["average_price_usd_per_lb"].pct_change() * 100
month_of_year = (
    monthly_prices.assign(month=monthly_prices.index.month)
    .groupby("month")["monthly_pct_change"]
    .agg(["mean", "median", "count"])
)
yearly_prices = copper["price_usd_per_lb"].resample("YE").mean().to_frame("average_price_usd_per_lb")
yearly_prices.index = yearly_prices.index.year

monthly_figure = px.line(
    monthly_prices.reset_index(),
    x="timestamp",
    y="average_price_usd_per_lb",
    title="Average copper futures price by month",
    labels={"timestamp": "Month", "average_price_usd_per_lb": "USD per pound"},
)
monthly_figure.show()

yearly_figure = px.bar(
    yearly_prices.reset_index(names="year"),
    x="year",
    y="average_price_usd_per_lb",
    title="Average copper futures price by year",
    labels={"average_price_usd_per_lb": "USD per pound"},
)
yearly_figure.show()

print("Typical monthly price change by calendar month")
display(month_of_year.round(3))

Typical monthly price change by calendar month


,mean,median,count
month,,,
1,1.736,2.384,26
2,2.052,1.745,26
3,2.057,2.363,26
4,2.176,-0.225,26
5,0.414,-0.387,26
6,-0.555,-0.870,26
7,1.031,1.677,26
8,-0.803,-2.002,26
9,0.441,0.149,27


### How to Read These Outputs

The first chart turns each month into one average price, which makes the long-run direction easier to see than daily data. The yearly bars compare average prices across calendar years. In the table, each row is a calendar month numbered `1` through `12`; `mean` and `median` are the typical percentage changes from the previous month, and `count` is how many historical examples were available. Treat a repeated pattern as a question to test, not a rule for future prices.

## Optional FRED Monthly Comparison

Turn on `LOAD_FRED_COMPARISON` only when the FRED cache already contains `PCOPPUSDM` or when `FRED_API_KEY` is set in the repository-root `.env`. This FRED series is monthly and uses USD per metric ton, so it is context for the daily Yahoo price rather than a second daily version of the same number.

FRED's standard adapter uses the observation date as an approximate availability date. That is acceptable for this visual exploration, but a later forecasting experiment must use a conservative publication-date rule before using delayed FRED data as a model input.

In [26]:
from aieng.forecasting.data.adapters import FREDAdapter


fred_copper: pd.DataFrame | None = None

if not LOAD_FRED_COMPARISON:
    print("FRED comparison is off. Set LOAD_FRED_COMPARISON = True to load PCOPPUSDM.")
else:
    try:
        fred_service = DataService()
        fred_service.register(
            FRED_COPPER_SERIES_ID,
            FREDAdapter(FRED_COPPER_ID, cache_dir=FRED_CACHE_DIR),
            SeriesMetadata(
                series_id=FRED_COPPER_SERIES_ID,
                description="Global copper price from FRED PCOPPUSDM",
                source="FRED (PCOPPUSDM)",
                units="USD per metric ton",
                frequency="MS",
            ),
        )
        fred_copper = fred_service.context(as_of=as_of_today).get_series(FRED_COPPER_SERIES_ID)
        print(f"FRED rows loaded: {len(fred_copper):,}")
        print(f"FRED coverage: {fred_copper['timestamp'].min().date()} to {fred_copper['timestamp'].max().date()}")
        print(f"Latest FRED value: ${fred_copper['value'].iloc[-1]:,.2f} per metric ton")
    except (RuntimeError, ValueError) as exc:
        print(f"FRED comparison was not loaded: {exc}")
        print(f"Add FRED_API_KEY to {REPO_ROOT / '.env'} and rerun with LOAD_FRED_COMPARISON = True.")

FRED rows loaded: 415
FRED coverage: 1992-01-01 to 2026-07-01
Latest FRED value: $13,542.82 per metric ton


### How to Read This Output

With the default setting, the message confirms that the optional FRED comparison was intentionally skipped. When enabled, the output reports the number of monthly observations, their date coverage, and the most recent FRED price in US dollars per metric ton. That value must not be compared directly with the Yahoo price in dollars per pound because the units differ.

In [27]:
if fred_copper is None:
    print("Skipping the Yahoo/FRED chart because the optional FRED series is unavailable.")
else:
    yahoo_monthly = copper["price_usd_per_lb"].resample("MS").mean()
    fred_monthly = (
        fred_copper.assign(timestamp=pd.to_datetime(fred_copper["timestamp"]))
        .set_index("timestamp")["value"]
        .resample("MS")
        .mean()
    )
    comparison_figure = make_subplots(specs=[[{"secondary_y": True}]])
    comparison_figure.add_trace(
        go.Scatter(x=yahoo_monthly.index, y=yahoo_monthly, name="Yahoo HG=F (USD/lb)"),
        secondary_y=False,
    )
    comparison_figure.add_trace(
        go.Scatter(x=fred_monthly.index, y=fred_monthly, name="FRED PCOPPUSDM (USD/metric ton)"),
        secondary_y=True,
    )
    comparison_figure.update_layout(title="Monthly copper prices: two sources, two unit systems")
    comparison_figure.update_yaxes(title_text="Yahoo Finance: USD per pound", secondary_y=False)
    comparison_figure.update_yaxes(title_text="FRED: USD per metric ton", secondary_y=True)
    comparison_figure.show()

### How to Read This Output

If FRED was unavailable, the message simply explains why no chart is shown. Otherwise, both lines show monthly averages over time, but use separate vertical axes: Yahoo `HG=F` is read from the left axis in US dollars per pound, and FRED `PCOPPUSDM` is read from the right axis in US dollars per metric ton. Compare whether their directions move together, not their raw heights.

## Related Daily Markets: Exploratory Correlations

Oil, the US dollar, and market volatility can move alongside copper. This section compares their *past daily changes* with copper's past daily changes. Correlation is only a pattern in the data; it does not prove that one market causes another.

In [28]:
import warnings

from aieng.forecasting.data.features import (
    StaticFrameAdapter,
    apply_one_business_day_feature_lag,
    to_log_return_feature,
)


COVARIATE_TICKERS = {
    "wti_oil_log_return_l1b": ("CL=F", "WTI oil"),
    "us_dollar_log_return_l1b": ("DX-Y.NYB", "US dollar index"),
    "vix_log_return_l1b": ("^VIX", "VIX market-volatility index"),
}

covariate_service = DataService()
covariate_frames: dict[str, pd.DataFrame] = {}

if not LOAD_YAHOO_COVARIATES:
    print("Yahoo covariates are off. Set LOAD_YAHOO_COVARIATES = True to load them.")
else:
    for series_id, (ticker, label) in COVARIATE_TICKERS.items():
        try:
            close_frame = YFinanceDailyAdapter(
                ticker,
                field="Adj Close",
                start=YAHOO_START,
                cache_dir=YAHOO_CACHE_DIR,
            ).fetch()
            feature_frame = apply_one_business_day_feature_lag(to_log_return_feature(close_frame))
            covariate_service.register(
                series_id,
                StaticFrameAdapter(feature_frame),
                SeriesMetadata(
                    series_id=series_id,
                    description=f"{label} close-to-close log return, lagged one business day",
                    source=f"Yahoo Finance ({ticker}), derived",
                    units="log return",
                    frequency="B",
                ),
            )
            covariate_frames[series_id] = feature_frame
        except (RuntimeError, ValueError, KeyError) as exc:
            warnings.warn(f"Skipping {label} ({ticker}): {exc}", stacklevel=1)

return_panel = copper[["daily_log_return"]].rename(columns={"daily_log_return": "copper_log_return"})
for series_id, feature_frame in covariate_frames.items():
    feature_series = feature_frame.set_index("timestamp")["value"].rename(series_id)
    return_panel = return_panel.join(feature_series, how="inner")

correlation_matrix = return_panel.corr()
print(f"Available covariates: {len(covariate_frames)}")
print(f"Rows used for joint correlation: {len(return_panel):,}")
display(correlation_matrix.round(3))

if correlation_matrix.shape[0] > 1:
    correlation_figure = px.imshow(
        correlation_matrix,
        text_auto=".2f",
        zmin=-1,
        zmax=1,
        color_continuous_scale="RdBu",
        title="Correlation of daily log returns",
    )
    correlation_figure.show()

Available covariates: 3
Rows used for joint correlation: 6,531


,copper_log_return,wti_oil_log_return_l1b,us_dollar_log_return_l1b,vix_log_return_l1b
copper_log_return,1.000,-0.012,-0.009,-0.066
wti_oil_log_return_l1b,-0.012,1.000,-0.135,-0.152
us_dollar_log_return_l1b,-0.009,-0.135,1.000,0.061
vix_log_return_l1b,-0.066,-0.152,0.061,1.000


### How to Read These Outputs

`Available covariates` reports how many of oil, the US dollar, and VIX loaded successfully. The row count is the number of dates shared by every loaded series. In the table and heat map, correlation ranges from `-1` to `1`: values near `1` mean the two daily changes usually move in the same direction, values near `-1` mean opposite directions, and values near `0` mean little straight-line relationship. The diagonal is always `1` because each series is perfectly related to itself. Correlation does not prove prediction or causation.

## Information Cutoff Check

A forecast made on a particular date must only use information known by that date. The repository enforces this with `released_at`: the daily close is visible one business day after its date. This cell proves the rule on a historical cutoff.

In [29]:
cutoff_as_of = pd.Timestamp(daily_history["timestamp"].max()) - pd.offsets.BDay(20)
visible_at_cutoff = daily_service.context(as_of=cutoff_as_of.to_pydatetime()).get_series(COPPER_SERIES_ID)

assert (visible_at_cutoff["released_at"] <= cutoff_as_of).all()
print(f"Full latest price date: {daily_history['timestamp'].max().date()}")
print(f"Chosen information cutoff: {cutoff_as_of.date()}")
print(f"Latest price visible at cutoff: {visible_at_cutoff['timestamp'].max().date()}")
print(f"Rows hidden from that historical view: {len(daily_history) - len(visible_at_cutoff):,}")
print("The close on the cutoff date itself is hidden until the next business day.")

Full latest price date: 2026-09-17
Chosen information cutoff: 2026-08-20
Latest price visible at cutoff: 2026-08-19
Rows hidden from that historical view: 20
The close on the cutoff date itself is hidden until the next business day.


### How to Read This Output

The full latest date is the newest data in the complete history. The cutoff date pretends that a forecast is being made 20 business days earlier. The latest visible price should be earlier than that cutoff because the close on the cutoff day is not released until the next business day. `Rows hidden` counts information that the historical forecast correctly cannot see.

## Forecast Questions for the Next Notebook

This notebook does not train a model. It only defines the two questions a later notebook will evaluate: the copper price 5 business days ahead and 21 business days ahead.

In [30]:
from aieng.forecasting.evaluation import ForecastingTask


forecast_tasks = [
    ForecastingTask(
        task_id="copper_price_5_business_days",
        target_series_id=COPPER_SERIES_ID,
        horizons=[5],
        frequency="B",
        payload_type="continuous",
        description="Forecast the Yahoo Finance HG=F copper futures close five business days ahead.",
    ),
    ForecastingTask(
        task_id="copper_price_21_business_days",
        target_series_id=COPPER_SERIES_ID,
        horizons=[21],
        frequency="B",
        payload_type="continuous",
        description="Forecast the Yahoo Finance HG=F copper futures close 21 business days ahead.",
    ),
]

task_overview = pd.DataFrame([
    {
        "Task": task.task_id,
        "Horizon": f"{task.horizons[0]} business days",
        "Meaning": "about one week" if task.horizons[0] == 5 else "about one month",
    }
    for task in forecast_tasks
])
display(task_overview)

,Task,Horizon,Meaning
0,copper_price_5_business_days,5 business days,about one week
1,copper_price_21_business_days,21 business days,about one month


### How to Read This Output

Each row is a future forecasting question, not a forecast result. The 5-business-day task asks for a price about one trading week ahead; the 21-business-day task asks for a price about one trading month ahead. A later notebook will score models separately for these two horizons.

## 9. Export Exploration Artifacts

The cleaned daily dataset and compact summaries are saved under the gitignored repository `data/` directory. The later forecasting notebook can read these artifacts without repeating exploratory cleaning steps.

In [31]:
artifact_dir = REPO_ROOT / "data" / "artifacts" / "copper_exploration"

if EXPORT_ARTIFACTS:
    artifact_dir.mkdir(parents=True, exist_ok=True)
    copper.reset_index().to_parquet(artifact_dir / "copper_daily_clean.parquet", index=False)
    price_summary.to_csv(artifact_dir / "price_summary.csv")
    annual_summary.to_csv(artifact_dir / "annual_price_summary.csv")
    month_of_year.to_csv(artifact_dir / "month_of_year_summary.csv")
    correlation_matrix.to_csv(artifact_dir / "daily_return_correlations.csv")
    print(f"Saved exploration artifacts to {artifact_dir}")
else:
    print("Set EXPORT_ARTIFACTS = True to save the cleaned dataset and summaries.")

Saved exploration artifacts to /home/coder/agentic-forecasting/data/artifacts/copper_exploration


### How to Read This Output

When export is enabled, this confirms the folder containing the cleaned daily data and CSV summaries. When it is disabled, no files were written; the message tells you which setting controls that behavior.

## What This Exploration Established

You now have a cleaned daily copper series, a view of its typical changes and larger swings, basic seasonal summaries, and a check that historical forecasts cannot see future closing prices.

The next notebook should compare forecasting models on the two defined horizons. It should keep the same target id and cutoff rule, score uncertainty as well as point accuracy, and avoid treating these exploratory charts as investment advice.